<a href="https://colab.research.google.com/github/Pshyam17/CMS-augmented-mamba/blob/main/Mamba_mentality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integration of continuous memory systems in revised deep learning models

## 1. Installing python packages

In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 162.8 MB/s eta 0:00:00


In [ ]:
!uv pip install datasets huggingface_hub lightning numpy pandas pyarrow torch tqdm wandb

Using Python 3.12.12 environment at: /usr
Resolved 61 packages in 389ms
Prepared 9 packages in 582ms
Installed 9 packages in 19ms
 + gitdb==4.0.12
 + gitpython==3.1.46
 + lightning==2.6.1
 + lightning-utilities==0.15.2
 + pytorch-lightning==2.6.1
 + sentry-sdk==2.52.0
 + smmap==5.0.2
 + torchmetrics==1.8.2
 + wandb==0.25.0


## 2. Importing Python libraries

In [ ]:
import pickle
import random
import sys
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

import lightning as pl
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

## 3. Defining all AI models to run on the benchmarks

In this section, we define with Python all machine learning models we are going to run in the benchmarks.

### 3.1 Seeing if GPU is avaiable or not

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

Using device: cpu



### 3.2 Defining all models that will act as baselines

In [ ]:
# CELL 3  — Baseline LSTM  (Section 3.2)

class BaselineLSTM(nn.Module):
    """Vanilla LSTM for sequence classification / matching."""
    def __init__(self, vocab_size, d_model=128, n_layers=2, n_classes=2,
                 max_len=2048, dropout=0.1, bidirectional=False, task='classification'):
        super().__init__()
        self.task = task
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        self.lstm = nn.LSTM(
            d_model, d_model, num_layers=n_layers,
            batch_first=True, dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        h_dim = d_model * (2 if bidirectional else 1)
        if task == 'retrieval':
            self.classifier = nn.Sequential(
                nn.Linear(h_dim * 2, d_model),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, n_classes),
            )
        else:
            self.classifier = nn.Sequential(
                nn.Linear(h_dim, d_model),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, n_classes),
            )
        self.dropout = nn.Dropout(dropout)

    def _encode(self, x):
        seq_len = x.size(1)
        h = self.embedding(x) + self.pos_encoding[:, :seq_len, :]
        h = self.dropout(h)
        out, _ = self.lstm(h)
        return out[:, -1, :]                       # last hidden state

    def forward(self, x, x2=None):
        if self.task == 'retrieval' and x2 is not None:
            h1 = self._encode(x)
            h2 = self._encode(x2)
            h = torch.cat([h1, h2], dim=-1)
        else:
            h = self._encode(x)
        return self.classifier(h)

In [ ]:
#Memory shape: (batch, n_slots, d_mem)

def __init__(self, d_model, n_slots=64, n_heads=4, dropout=0.1):
    super().__init__()
    self.n_slots = n_slots
    self.d_model = d_model
    self.n_heads = n_heads
    assert d_model % n_heads == 0
    self.head_dim = d_model // n_heads

    # Read head — multi-head content-based attention
    self.q_proj = nn.Linear(d_model, d_model)
    self.k_proj = nn.Linear(d_model, d_model)
    self.v_proj = nn.Linear(d_model, d_model)
    self.read_out = nn.Linear(d_model, d_model)

    # Write head — gated update
    self.write_gate = nn.Sequential(
        nn.Linear(d_model * 2, d_model),
        nn.Sigmoid(),
    )
    self.write_content = nn.Linear(d_model, d_model)
    self.erase_gate = nn.Sequential(
        nn.Linear(d_model, d_model),
        nn.Sigmoid(),
    )

    # Addressing — sharpened softmax over slots
    self.address_proj = nn.Linear(d_model, d_model)
    self.sharpening_beta = nn.Parameter(torch.ones(1))

    self.layer_norm = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

def _init_memory(self, batch_size, device):
    """Initialise memory slots with small random values."""
    return torch.randn(batch_size, self.n_slots, self.d_model,
                       device=device) * 0.01

def _address(self, query, memory):
    """Content-based addressing: softmax similarity to memory keys."""
    # query: (B, D)  memory: (B, S, D)
    k = self.address_proj(memory)                          # (B, S, D)
    score = torch.einsum('bd,bsd->bs', query, k)          # (B, S)
    score = score / (self.d_model ** 0.5)
    beta = F.softplus(self.sharpening_beta)
    weights = F.softmax(beta * score, dim=-1)              # (B, S)
    return weights

def read(self, h, memory):
    """Multi-head content-based read from memory."""
    B = h.size(0)
    q = self.q_proj(h).view(B, 1, self.n_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(memory).view(B, self.n_slots, self.n_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(memory).view(B, self.n_slots, self.n_heads, self.head_dim).transpose(1, 2)

    attn = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
    attn = F.softmax(attn, dim=-1)
    attn = self.dropout(attn)
    read_vec = torch.matmul(attn, v)                      # (B, H, 1, Dh)
    read_vec = read_vec.transpose(1, 2).reshape(B, self.d_model)
    return self.read_out(read_vec)

def write(self, h, memory):
    """Gated erase-then-add write to memory."""
    weights = self._address(h, memory).unsqueeze(-1)       # (B, S, 1)
    erase = self.erase_gate(h).unsqueeze(1)                # (B, 1, D)
    add = self.write_content(h).unsqueeze(1)               # (B, 1, D)
    memory = memory * (1 - weights * erase) + weights * add
    return memory

def forward(self, h, memory):
    # h      : current hidden state  (B, D)
    # memory : memory bank            (B, S, D)
    # Returns: (read_vector, updated_memory)

    read_vec = self.read(h, memory)
    # Gate how much of the read vector to incorporate
    gate = self.write_gate(torch.cat([h, read_vec], dim=-1))
    read_vec = gate * read_vec
    read_vec = self.layer_norm(read_vec)
    # Write current hidden state into memory
    memory = self.write(h, memory)
    return read_vec, memory

### 3.3 Defining all models we are researching to try on the benchmarks

## 4. Long Range Arena benchmark implemented from strach

The deep learning model requires independent hyperparameter tuning runs across all six benchmarks.

Testing one optimal model against four optimal baselines results in 30 total runs (6 runs × 5 models).

Each run includes hyperparameter tuning and metric-based verification.

Benchmarks are implemented from scratch based on Long Range Arena: A Benchmark for Efficient Transformers.

Link: https://arxiv.org/abs/2011.04006

### 4.1 ListOps: Hierarchical parsing of mathematical expressions

 Can the model understand nested structure, like matching parentheses and applying operations in the right order?

In [ ]:
class ListOpsGenerator:
    def __init__(self, max_depth=10, max_args=10, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        self.max_depth = max_depth
        self.max_args = max_args
        self.ops = ['MAX', 'MIN', 'MED', 'SM']
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        self.vocab = {'[PAD]': 0, '[': 1, ']': 2, 'MAX': 3, 'MIN': 4,
                      'MED': 5, 'SM': 6}
        for i in range(10):
            self.vocab[str(i)] = 7 + i
        self.vocab_size = len(self.vocab)
        self._token_pattern = re.compile(r'\[|\]|MAX|MIN|MED|SM|\d')

    def _eval(self, expr):
        expr = expr.strip()
        if expr.isdigit():
            return int(expr)
        expr = expr[1:-1].strip()
        parts = expr.split(None, 1)
        op = parts[0]
        if len(parts) == 1:
            return 0

        args, current, depth = [], [], 0
        for c in parts[1]:
            if c == '[':
                depth += 1
                current.append(c)
            elif c == ']':
                depth -= 1
                current.append(c)
            elif c == ' ' and depth == 0:
                if current:
                    args.append(''.join(current))
                    current = []
            else:
                current.append(c)
        if current:
            args.append(''.join(current))

        vals = [self._eval(arg) for arg in args]
        if not vals:
            return 0
        if op == 'MAX':
            return max(vals)
        elif op == 'MIN':
            return min(vals)
        elif op == 'MED':
            s = sorted(vals)
            n = len(s)
            return s[n//2] if n % 2 else (s[n//2-1] + s[n//2]) // 2
        else:
            return sum(vals) % 10

    def _gen_expr(self, depth=0, curr_len=0, target_len=1000):
        if curr_len > target_len * 1.5:
            term_prob = 0.8
        elif curr_len > target_len:
            term_prob = 0.5
        elif curr_len < target_len * 0.3 and depth < self.max_depth - 2:
            term_prob = 0.1
        else:
            term_prob = 0.25

        if depth >= self.max_depth or (depth > 0 and random.random() < term_prob):
            return str(random.randint(0, 9))

        op = random.choice(self.ops)
        num_args = random.randint(3, self.max_args) if curr_len < target_len * 0.5 else random.randint(2, 5)
        args = []
        for _ in range(num_args):
            arg = self._gen_expr(depth + 1, curr_len + len(args) * 10, target_len)
            args.append(arg)
            curr_len += len(arg)
        return f"[ {op} {' '.join(args)} ]"

    def tokenize(self, expr):
        return [self.vocab[t] for t in self._token_pattern.findall(expr)]

    def _generate_single(self, target_len, min_len, max_len):
        for _ in range(100):
            expr = self._gen_expr(target_len=target_len)
            if min_len <= len(expr) <= max_len:
                try:
                    return (self.tokenize(expr), self._eval(expr))
                except:
                    pass
        return None

    def generate(self, n, min_len=500, max_len=2000, num_workers=8):
        data, target_len = [], (min_len + max_len) // 2
        pbar = tqdm(total=n, desc="Generating samples")

        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = {executor.submit(self._generate_single, target_len, min_len, max_len): i
                       for i in range(min(n * 2, 1000))}
            while len(data) < n:
                for future in as_completed(futures):
                    if len(data) >= n:
                        break
                    result = future.result()
                    if result:
                        data.append(result)
                        pbar.update(1)
                    del futures[future]
                needed = n - len(data) - len(futures)
                if needed > 0:
                    for _ in range(min(needed * 2, 100)):
                        futures[executor.submit(self._generate_single, target_len, min_len, max_len)] = len(futures)
        pbar.close()
        return data[:n]

In [ ]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Google Drive already mounted.")

# Define the directory in Google Drive
drive_dir = "/content/drive/MyDrive/LRA_Datasets"
os.makedirs(drive_dir, exist_ok=True)
print(f"Ensured directory exists: {drive_dir}")

gen = ListOpsGenerator(seed=42)

print("Generating ListOps dataset...\n")
print("Training set (96000 samples):")
train_data = gen.generate(9600)

print("\nValidation set (2000 samples):")
val_data = gen.generate(2000)

print("\nTest set (2000 samples):")
test_data = gen.generate(2000)

# Create datasets
train_ds = ListOpsDataset(train_data)
val_ds = ListOpsDataset(val_data)
test_ds = ListOpsDataset(test_data)

# Save to Google Drive
listops_save_path = os.path.join(drive_dir, 'listops.pt')
torch.save({'train': train_ds, 'val': val_ds, 'test': test_ds,
            'vocab_size': gen.vocab_size}, listops_save_path)

print(f"\nSaved ListOps dataset to: {listops_save_path}")
print(f"Vocab size: {gen.vocab_size}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Mounted at /content/drive
Google Drive mounted.
Ensured directory exists: /content/drive/MyDrive/LRA_Datasets
Generating ListOps dataset...

Training set (96000 samples):



Generating samples: 100%|██████████| 9600/9600 [01:35<00:00, 100.99it/s]



Validation set (2000 samples):



Generating samples: 100%|██████████| 2000/2000 [00:19<00:00, 102.32it/s]



Test set (2000 samples):



Generating samples: 100%|██████████| 2000/2000 [00:19<00:00, 101.80it/s]



Saved ListOps dataset to: /content/drive/MyDrive/LRA_Datasets/listops.pt
Vocab size: 17
Train: 9600 | Val: 2000 | Test: 2000


### 4.2 Text: Sentiment classification on IMDb reviews

1.   List item
2.   List item



Can the model figure out sentiment when reading raw characters instead of words across a long document?

In [ ]:
class IMDbGenerator:
    """
    IMDb byte-level text classification for Long Range Arena benchmark.
    - Binary classification (positive/negative sentiment)
    - Character/byte-level tokenization (vocab size 256)
    - Max sequence length: 4096 (as per LRA spec)
    """
    def __init__(self, max_len=4096, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        self.max_len = max_len
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Byte-level vocabulary (256 possible byte values + PAD)
        self.vocab_size = 257  # 0 = PAD, 1-256 = byte values
        self.pad_token = 0

    def tokenize(self, text):
        """Convert text to byte-level tokens (1-256, 0 reserved for PAD)."""
        # Encode to bytes and shift by 1 (so 0 is reserved for padding)
        return [b + 1 for b in text.encode('utf-8', errors='replace')]

    def _process_single(self, example):
        """Process a single IMDb example."""
        text = example['text']
        label = example['label']  # 0 = negative, 1 = positive
        tokens = self.tokenize(text)
        return (tokens, label)

    def load_and_process(self, num_workers=8):
        """Load IMDb dataset and process to byte-level tokens."""
        print("Loading IMDb dataset from HuggingFace...")
        dataset = load_dataset('imdb')

        train_data = []
        test_data = []

        # Process training set
        print("\nProcessing training set (25000 samples):")
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(self._process_single, ex) for ex in dataset['train']]
            for future in tqdm(as_completed(futures), total=len(futures), desc="Train"):
                train_data.append(future.result())

        # Process test set
        print("\nProcessing test set (25000 samples):")
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(self._process_single, ex) for ex in dataset['test']]
            for future in tqdm(as_completed(futures), total=len(futures), desc="Test"):
                test_data.append(future.result())

        # Shuffle training data
        random.shuffle(train_data)

        # Split train into train/val (LRA uses 25k train, but we create a val set)
        val_size = 5000
        val_data = train_data[:val_size]
        train_data = train_data[val_size:]

        return train_data, val_data, test_data

In [ ]:
class IMDbDataset(Dataset):
    """PyTorch Dataset for byte-level IMDb classification."""
    def __init__(self, data, max_len=4096):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad to max_len
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Google Drive already mounted.")

# Define the directory in Google Drive
drive_dir = "/content/drive/MyDrive/LRA_Datasets"
os.makedirs(drive_dir, exist_ok=True)
print(f"Ensured directory exists: {drive_dir}")

gen = IMDbGenerator(max_len=4096, seed=42, device=device)
print(f"Using device: {gen.device}")
print(f"Vocab size: {gen.vocab_size} (byte-level)")
print(f"Max sequence length: {gen.max_len}\n")

# Load and process IMDb dataset
train_data, val_data, test_data = gen.load_and_process(num_workers=8)

# Create PyTorch datasets
train_ds = IMDbDataset(train_data, max_len=4096)
val_ds = IMDbDataset(val_data, max_len=4096)
test_ds = IMDbDataset(test_data, max_len=4096)

# Save to Google Drive
imdb_save_path = os.path.join(drive_dir, 'imdb_lra.pt')
torch.save({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds,
    'vocab_size': gen.vocab_size,
    'max_len': gen.max_len,
    'num_classes': 2
}, imdb_save_path)

print(f"\nSaved to {imdb_save_path}")
print(f"Vocab size: {gen.vocab_size}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Google Drive already mounted.
Ensured directory exists: /content/drive/MyDrive/LRA_Datasets
Using device: cuda
Vocab size: 257 (byte-level)
Max sequence length: 4096

Loading IMDb dataset from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...


Processing training set (25000 samples):



Train: 100%|██████████| 25000/25000 [00:00<00:00, 210113.58it/s]



Processing test set (25000 samples):



Test: 100%|██████████| 25000/25000 [00:00<00:00, 211176.54it/s]



Saved to /content/drive/MyDrive/LRA_Datasets/imdb_lra.pt
Vocab size: 257
Train: 20000 | Val: 5000 | Test: 25000


### 4.3 Retrieval: Document similarity matching (ACL Anthology)


Can the model tell if two documents are related by comparing their full contents?

Link: https://huggingface.co/datasets/WINGNUS/ACL-OCL

In [ ]:
class ACLRetrievalGenerator:
    """
    ACL Anthology document retrieval for Long Range Arena benchmark.
    - Binary classification: are two papers similar (cited) or not?
    - Byte-level tokenization (vocab size 257)
    - Max sequence length: 4000 per document (8000 total for pair)
    - Similarity defined by citation relationship
    """
    def __init__(self, max_len_per_doc=4000, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        self.max_len_per_doc = max_len_per_doc
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Byte-level vocabulary (same as IMDb for consistency)
        self.vocab_size = 257  # 0 = PAD, 1-256 = byte values
        self.pad_token = 0
        self.sep_token = 256  # Special separator between documents

    def tokenize(self, text):
        """Convert text to byte-level tokens."""
        if text is None or (isinstance(text, float) and np.isnan(text)):
            return []
        text = str(text)
        return [b + 1 for b in text.encode('utf-8', errors='replace')]

    def load_dataset(self):
        """Load ACL-OCL dataset from HuggingFace."""
        print("Loading ACL-OCL dataset from HuggingFace...")
        print("Dataset: WINGNUS/ACL-OCL")

        import pandas as pd
        from huggingface_hub import hf_hub_download

        # Download the parquet file from HuggingFace Hub
        # Note: filename is "acl-publication-info.74k.v2.parquet" (with v2)
        print("Downloading paper metadata (515 MB)...")
        parquet_path = hf_hub_download(
            repo_id="WINGNUS/ACL-OCL",
            filename="acl-publication-info.74k.v2.parquet",
            repo_type="dataset"
        )

        # Also download citation graph for true similarity pairs
        print("Downloading citation graph (73 MB)...")
        citations_path = hf_hub_download(
            repo_id="WINGNUS/ACL-OCL",
            filename="acl_full_citations.parquet",
            repo_type="dataset"
        )

        # Load with pandas
        print("Loading parquet files...")
        df = pd.read_parquet(parquet_path)
        citations_df = pd.read_parquet(citations_path)

        print(f"Loaded {len(df)} papers")
        print(f"Loaded {len(citations_df)} citation relationships")

        return df, citations_df

    def build_citation_graph(self, df, citations_df):
        """
        Build citation relationships from the citation graph.
        Papers that cite each other are considered similar.
        """
        print("Building citation-based similarity pairs...")

        # Filter papers with valid abstracts and full text
        df_valid = df[
            (df['abstract'].notna()) &
            (df['abstract'].str.len() > 100) &
            (df['full_text'].notna()) &
            (df['full_text'].str.len() > 200)
        ].copy()

        print(f"Papers with valid text: {len(df_valid)}")

        # Create paper index mapping acl_id to index
        papers = df_valid[['acl_id', 'abstract', 'full_text', 'title']].reset_index(drop=True)
        paper_id_to_idx = {pid: idx for idx, pid in enumerate(papers['acl_id'].tolist())}

        # Build citation pairs from citations_df
        # The citation df should have citing and cited paper IDs
        print(f"Citation graph columns: {citations_df.columns.tolist()}")

        return papers, paper_id_to_idx, citations_df

    def create_pairs(self, papers, paper_id_to_idx, citations_df, n_pairs, positive_ratio=0.5):
        """
        Create document pairs for retrieval task.
        - Positive pairs: papers with citation relationship
        - Negative pairs: random paper pairs without citation relationship
        """
        print(f"Creating {n_pairs} document pairs...")

        pairs = []
        n_positive = int(n_pairs * positive_ratio)
        n_negative = n_pairs - n_positive

        papers_list = papers.to_dict('records')
        n_papers = len(papers_list)
        valid_acl_ids = set(papers['acl_id'].tolist())

        # Try to use citation relationships for positive pairs
        # First, let's see what the citation df looks like
        citation_pairs = set()
        try:
            # Common column names for citation data
            for col_pair in [('citing', 'cited'), ('source', 'target'),
                            ('paper_id', 'cited_paper_id'), ('from', 'to')]:
                if col_pair[0] in citations_df.columns and col_pair[1] in citations_df.columns:
                    for _, row in citations_df.iterrows():
                        citing = str(row[col_pair[0]])
                        cited = str(row[col_pair[1]])
                        if citing in valid_acl_ids and cited in valid_acl_ids:
                            citation_pairs.add((citing, cited))
                    break

            # If column detection failed, try first two columns
            if len(citation_pairs) == 0 and len(citations_df.columns) >= 2:
                col1, col2 = citations_df.columns[0], citations_df.columns[1]
                for _, row in citations_df.head(10000).iterrows():
                    citing = str(row[col1])
                    cited = str(row[col2])
                    if citing in valid_acl_ids and cited in valid_acl_ids:
                        citation_pairs.add((citing, cited))
        except Exception as e:
            print(f"Warning: Could not parse citations: {e}")

        print(f"Found {len(citation_pairs)} valid citation pairs")

        # Create positive pairs from citations (or fallback to proximity)
        print(f"  Creating {n_positive} positive pairs...")
        citation_list = list(citation_pairs)

        for i in tqdm(range(n_positive), desc="Positive pairs"):
            if citation_list and i < len(citation_list):
                # Use actual citation relationship
                citing_id, cited_id = citation_list[i % len(citation_list)]
                idx1 = paper_id_to_idx[citing_id]
                idx2 = paper_id_to_idx[cited_id]
            else:
                # Fallback: use nearby papers (same venue/time proxy)
                idx1 = random.randint(0, n_papers - 50)
                offset = random.randint(1, min(49, n_papers - idx1 - 1))
                idx2 = idx1 + offset

            doc1_text = str(papers_list[idx1].get('abstract', '')) + " " + str(papers_list[idx1].get('full_text', ''))[:2000]
            doc2_text = str(papers_list[idx2].get('abstract', '')) + " " + str(papers_list[idx2].get('full_text', ''))[:2000]

            pairs.append((doc1_text, doc2_text, 1))  # label 1 = similar

        # Create negative pairs (random dissimilar papers)
        print(f"  Creating {n_negative} negative pairs...")
        for _ in tqdm(range(n_negative), desc="Negative pairs"):
            idx1 = random.randint(0, n_papers - 1)
            # Pick a distant paper (at least 1000 papers apart)
            idx2 = (idx1 + random.randint(1000, n_papers - 1)) % n_papers

            doc1_text = str(papers_list[idx1].get('abstract', '')) + " " + str(papers_list[idx1].get('full_text', ''))[:2000]
            doc2_text = str(papers_list[idx2].get('abstract', '')) + " " + str(papers_list[idx2].get('full_text', ''))[:2000]

            pairs.append((doc1_text, doc2_text, 0))  # label 0 = dissimilar

        random.shuffle(pairs)
        return pairs

    def process_pairs(self, pairs):
        """Convert document pairs to tokenized format."""
        print("Tokenizing document pairs...")
        processed = []

        for doc1, doc2, label in tqdm(pairs, desc="Tokenizing"):
            tokens1 = self.tokenize(doc1)[:self.max_len_per_doc]
            tokens2 = self.tokenize(doc2)[:self.max_len_per_doc]

            # Concatenate: [doc1] [SEP] [doc2]
            combined = tokens1 + [self.sep_token] + tokens2
            processed.append((combined, label))

        return processed


In [ ]:
class ACLRetrievalDataset(Dataset):
    """PyTorch Dataset for ACL document retrieval."""
    def __init__(self, data, max_len=8001):  # 4000 + 1 (sep) + 4000
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Google Drive already mounted.")

# Define the directory in Google Drive
drive_dir = "/content/drive/MyDrive/LRA_Datasets"
os.makedirs(drive_dir, exist_ok=True)
print(f"Ensured directory exists: {drive_dir}")

# Initialize generator
gen = ACLRetrievalGenerator(max_len_per_doc=4000, seed=42, device=device)
print(f"Using device: {gen.device}")
print(f"Vocab size: {gen.vocab_size} (byte-level)")
print(f"Max length per document: {gen.max_len_per_doc}")
print(f"Total max sequence length: {gen.max_len_per_doc * 2 + 1}\n")

# Load ACL-OCL dataset (both papers and citations)
df, citations_df = gen.load_dataset()

# Build paper index with citation graph
papers, paper_id_to_idx, citations_df = gen.build_citation_graph(df, citations_df)

# Create train/val/test splits
print("\n" + "="*50)
print("Creating dataset splits...")
print("="*50)

train_pairs = gen.create_pairs(papers, paper_id_to_idx, citations_df, n_pairs=20000)
val_pairs = gen.create_pairs(papers, paper_id_to_idx, citations_df, n_pairs=2000)
test_pairs = gen.create_pairs(papers, paper_id_to_idx, citations_df, n_pairs=2000)

# Process pairs
train_data = gen.process_pairs(train_pairs)
val_data = gen.process_pairs(val_pairs)
test_data = gen.process_pairs(test_pairs)

# Create PyTorch datasets
max_seq_len = gen.max_len_per_doc * 2 + 1  # doc1 + sep + doc2
train_ds = ACLRetrievalDataset(train_data, max_len=max_seq_len)
val_ds = ACLRetrievalDataset(val_data, max_len=max_seq_len)
test_ds = ACLRetrievalDataset(test_data, max_len=max_seq_len)

# Save to Google Drive
acl_retrieval_save_path = os.path.join(drive_dir, 'acl_retrieval_lra.pt')
torch.save({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds,
    'vocab_size': gen.vocab_size,
    'max_len': max_seq_len,
    'num_classes': 2,
    'task': 'retrieval'
}, acl_retrieval_save_path)

print(f"\n" + "="*50)
print("DATASET SAVED SUCCESSFULLY")
print("="*50)
print(f"Saved to: {acl_retrieval_save_path}")
print(f"Vocab size: {gen.vocab_size}")
print(f"Max sequence length: {max_seq_len}")
print(f"Number of classes: 2 (similar/dissimilar)")
print(f"\nSplit sizes:")
print(f"  Train: {len(train_ds):,} pairs")
print(f"  Val:   {len(val_ds):,} pairs")
print(f"  Test:  {len(test_ds):,} pairs")

# Verify a sample
print(f"\nSample verification:")
sample = train_ds[0]
print(f"  Input shape: {sample['input_ids'].shape}")
print(f"  Mask shape:  {sample['attention_mask'].shape}")
print(f"  Label:       {sample['labels'].item()}")

Mounted at /content/drive
Google Drive mounted.
Ensured directory exists: /content/drive/MyDrive/LRA_Datasets
Using device: cpu
Vocab size: 257 (byte-level)
Max length per document: 4000
Total max sequence length: 8001

Loading ACL-OCL dataset from HuggingFace...
Dataset: WINGNUS/ACL-OCL


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


acl-publication-info.74k.v2.parquet:   0%|          | 0.00/515M [00:00<?, ?B/s]

acl_full_citations.parquet:   0%|          | 0.00/73.4M [00:00<?, ?B/s]

Loading parquet files...
Loaded 73285 papers
Loaded 3812256 citation relationships
Building citation-based similarity pairs...
Papers with valid text: 63758
Citation graph columns: ['id', 'citingpaperid', 'citedpaperid', 'is_citedpaperid_acl', 'is_citingpaperid_acl']

Creating dataset splits...
Creating 20000 document pairs...
Found 0 valid citation pairs
  Creating 10000 positive pairs...


...

Tokenizing document pairs...


Tokenizing: 100%|██████████| 2000/2000 [00:00<00:00, 8064.94it/s]


Tokenizing document pairs...


Tokenizing: 100%|██████████| 2000/2000 [00:00<00:00, 6664.34it/s]



DATASET SAVED SUCCESSFULLY
Saved to: /content/drive/MyDrive/LRA_Datasets/acl_retrieval_lra.pt
Vocab size: 257
Max sequence length: 8001
Number of classes: 2 (similar/dissimilar)

Split sizes:
  Train: 20,000 pairs
  Val:   2,000 pairs
  Test:  2,000 pairs

Sample verification:
  Input shape: torch.Size([8001])
  Mask shape:  torch.Size([8001])
  Label:       0


### 4.4 Image: Sequential CIFAR-10 classification (pixels as sequence)

Can the model recognize an image when pixels are fed one at a time in a long line?

In [ ]:
class CIFAR10SequentialGenerator:
    """
    CIFAR-10 sequential image classification for Long Range Arena benchmark.
    - 10-class classification
    - Images flattened to 1D sequence: 32x32x3 = 3072 tokens
    - Each pixel channel is a token (grayscale version: 32x32 = 1024 tokens)
    - Vocab size: 256 (pixel values 0-255)
    """
    def __init__(self, grayscale=True, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        self.grayscale = grayscale
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Pixel vocabulary (0-255) + PAD token
        self.vocab_size = 257  # 0 = PAD, 1-256 = pixel values (shifted by 1)
        self.pad_token = 0

        # Sequence length depends on grayscale vs color
        # LRA uses grayscale: 32x32 = 1024
        # Color would be: 32x32x3 = 3072
        self.seq_len = 1024 if grayscale else 3072
        self.num_classes = 10

        # CIFAR-10 class names
        self.class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                           'dog', 'frog', 'horse', 'ship', 'truck']

    def load_dataset(self):
        """Load CIFAR-10 dataset from HuggingFace."""
        print("Loading CIFAR-10 dataset from HuggingFace...")

        dataset = load_dataset('cifar10')

        print(f"Train samples: {len(dataset['train'])}")
        print(f"Test samples: {len(dataset['test'])}")

        return dataset

    def process_image(self, image):
        """
        Convert PIL image to sequential tokens.
        - Convert to grayscale if specified
        - Flatten to 1D sequence
        - Shift pixel values by 1 (so 0 can be PAD)
        """
        # Convert PIL to numpy
        img_array = np.array(image)

        if self.grayscale:
            # Convert RGB to grayscale: 0.299*R + 0.587*G + 0.114*B
            if len(img_array.shape) == 3:
                img_array = np.dot(img_array[..., :3], [0.299, 0.587, 0.114])
            img_array = img_array.astype(np.uint8)

        # Flatten to 1D sequence
        flat = img_array.flatten()

        # Shift by 1 so 0 is reserved for PAD
        tokens = [int(p) + 1 for p in flat]

        return tokens

    def process_dataset(self, dataset, split='train', num_workers=8):
        """Process entire split of CIFAR-10 to sequential format."""
        print(f"\nProcessing {split} split...")

        data = []
        examples = dataset[split]

        for i in tqdm(range(len(examples)), desc=f"Processing {split}"):
            image = examples[i]['img']
            label = examples[i]['label']

            tokens = self.process_image(image)
            data.append((tokens, label))

        return data

In [ ]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Google Drive already mounted.")

# Define the directory in Google Drive
drive_dir = "/content/drive/MyDrive/LRA_Datasets"
os.makedirs(drive_dir, exist_ok=True)
print(f"Ensured directory exists: {drive_dir}")

gen = CIFAR10SequentialGenerator(grayscale=True, seed=42, device=device)
print(f"Using device: {gen.device}")
print(f"Vocab size: {gen.vocab_size} (pixel values)")
print(f"Sequence length: {gen.seq_len} ({'grayscale 32x32' if gen.grayscale else 'color 32x32x3'})")
print(f"Number of classes: {gen.num_classes}")
print(f"Classes: {gen.class_names}\n")

# Load CIFAR-10 dataset
dataset = gen.load_dataset()

# Process train and test sets
train_data = gen.process_dataset(dataset, split='train')
test_data = gen.process_dataset(dataset, split='test')

# Create validation split from training data (LRA uses 45k train, 5k val)
random.shuffle(train_data)
val_size = 5000
val_data = train_data[:val_size]
train_data = train_data[val_size:]

print(f"\nSplit sizes after validation split:")
print(f"  Train: {len(train_data)}")
print(f"  Val:   {len(val_data)}")
print(f"  Test:  {len(test_data)}")

# Create PyTorch datasets
train_ds = CIFAR10SequentialDataset(train_data, seq_len=gen.seq_len)
val_ds = CIFAR10SequentialDataset(val_data, seq_len=gen.seq_len)
test_ds = CIFAR10SequentialDataset(test_data, seq_len=gen.seq_len)

# Save to Google Drive
cifar10_save_path = os.path.join(drive_dir, 'cifar10_sequential_lra.pt')
torch.save({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds,
    'vocab_size': gen.vocab_size,
    'seq_len': gen.seq_len,
    'num_classes': gen.num_classes,
    'class_names': gen.class_names,
    'grayscale': gen.grayscale,
    'task': 'image_classification'
}, cifar10_save_path)

print(f"\n" + "="*50)
print("DATASET SAVED SUCCESSFULLY")
print("="*50)
print(f"Saved to: {cifar10_save_path}")
print(f"Vocab size: {gen.vocab_size}")
print(f"Sequence length: {gen.seq_len}")
print(f"Number of classes: {gen.num_classes}")
print(f"\nSplit sizes:")
print(f"  Train: {len(train_ds):,} images")
print(f"  Val:   {len(val_ds):,} images")
print(f"  Test:  {len(test_ds):,} images")

Google Drive already mounted.
Ensured directory exists: /content/drive/MyDrive/LRA_Datasets
Using device: cpu
Vocab size: 257 (pixel values)
Sequence length: 1024 (grayscale 32x32)
Number of classes: 10
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Loading CIFAR-10 dataset from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Train samples: 50000
Test samples: 10000

Processing train split...


Processing train: 100%|██████████| 50000/50000 [00:16<00:00, 3021.16it/s]



Processing test split...


Processing test: 100%|██████████| 10000/10000 [00:03<00:00, 3040.44it/s]



Split sizes after validation split:
  Train: 45000
  Val:   5000
  Test:  10000

DATASET SAVED SUCCESSFULLY
Saved to: /content/drive/MyDrive/LRA_Datasets/cifar10_sequential_lra.pt
Vocab size: 257
Sequence length: 1024
Number of classes: 10

Split sizes:
  Train: 45,000 images
  Val:   5,000 images
  Test:  10,000 images


### 4.5 Two Pathfinder datasets

Defining a common class for both

In [ ]:
class PathfinderGenerator:
    """
    Base Pathfinder generator for Long Range Arena benchmark.

    Task: Binary classification - are two dots connected by a path?
    - Pathfinder: 32x32 images -> 1024 sequence length
    - Path-X: 128x128 images -> 16384 sequence length

    The images contain:
    - Two marked endpoints (dots)
    - Curvy lines (some connecting the dots, some distractors)
    - Background noise

    Label: 1 if dots are connected, 0 if not
    """
    def __init__(self, resolution=32, seed=42, device=None,
                 contour_length=14, num_distractor_snakes=6, snake_contrast=0.5):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        self.resolution = resolution
        self.seq_len = resolution * resolution  # 32x32=1024 or 128x128=16384
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Snake/path parameters
        self.contour_length = contour_length
        self.num_distractor_snakes = num_distractor_snakes
        self.snake_contrast = snake_contrast

        # Vocab: grayscale pixel values (0-255) + PAD
        self.vocab_size = 257
        self.pad_token = 0
        self.num_classes = 2  # connected or not

    def _draw_snake(self, img, start_pos, length, mark_endpoints=False):
        """Draw a curvy snake/path on the image."""
        h, w = img.shape
        x, y = start_pos
        positions = [(x, y)]

        # Random walk to create curvy path
        angle = random.uniform(0, 2 * np.pi)

        for _ in range(length):
            # Add some curvature
            angle += random.gauss(0, 0.5)

            # Calculate new position
            dx = int(round(np.cos(angle) * 2))
            dy = int(round(np.sin(angle) * 2))

            new_x = max(2, min(w - 3, x + dx))
            new_y = max(2, min(h - 3, y + dy))

            # Draw the segment
            x, y = new_x, new_y
            positions.append((x, y))

            # Draw with some thickness
            for ox in range(-1, 2):
                for oy in range(-1, 2):
                    px, py = x + ox, y + oy
                    if 0 <= px < w and 0 <= py < h:
                        img[py, px] = min(255, img[py, px] + int(128 * self.snake_contrast))

        # Mark endpoints if requested
        if mark_endpoints and len(positions) >= 2:
            # Draw circles at start and end
            for pos in [positions[0], positions[-1]]:
                cx, cy = pos
                for ox in range(-2, 3):
                    for oy in range(-2, 3):
                        if ox*ox + oy*oy <= 4:  # circle
                            px, py = cx + ox, cy + oy
                            if 0 <= px < w and 0 <= py < h:
                                img[py, px] = 255

        return positions[0], positions[-1] if len(positions) > 1 else positions[0]

    def _generate_sample(self, connected=True):
        """Generate a single pathfinder sample."""
        h, w = self.resolution, self.resolution

        # Start with noisy background
        img = np.random.randint(0, 50, (h, w), dtype=np.uint8)

        # Random start position for main path
        margin = max(5, self.resolution // 8)
        start_x = random.randint(margin, w - margin)
        start_y = random.randint(margin, h - margin)

        # Draw the main path (the one we're asking about)
        start_pos, end_pos = self._draw_snake(
            img, (start_x, start_y),
            self.contour_length,
            mark_endpoints=True
        )

        # Draw distractor snakes
        for _ in range(self.num_distractor_snakes):
            dist_x = random.randint(margin, w - margin)
            dist_y = random.randint(margin, h - margin)
            dist_length = random.randint(self.contour_length // 2, self.contour_length)
            self._draw_snake(img, (dist_x, dist_y), dist_length, mark_endpoints=False)

        # If not connected, we need to break the path or create separate endpoints
        if not connected:
            # Redraw with separate, unconnected endpoint markers
            # Add a second pair of dots that aren't connected
            new_x = random.randint(margin, w - margin)
            new_y = random.randint(margin, h - margin)
            # Make sure it's far from original
            while abs(new_x - start_x) < w // 3 and abs(new_y - start_y) < h // 3:
                new_x = random.randint(margin, w - margin)
                new_y = random.randint(margin, h - margin)

            # Draw endpoint marker at new location (not connected)
            for ox in range(-2, 3):
                for oy in range(-2, 3):
                    if ox*ox + oy*oy <= 4:
                        px, py = new_x + ox, new_y + oy
                        if 0 <= px < w and 0 <= py < h:
                            img[py, px] = 255

        # Flatten and convert to tokens (shift by 1 for PAD)
        tokens = [int(p) + 1 for p in img.flatten()]
        label = 1 if connected else 0

        return tokens, label

    def generate(self, n_samples, num_workers=1):
        """Generate n_samples of pathfinder data."""
        print(f"Generating {n_samples} samples (resolution={self.resolution}x{self.resolution})...")

        data = []
        for i in tqdm(range(n_samples), desc=f"Generating"):
            connected = random.random() < 0.5  # 50/50 balance
            tokens, label = self._generate_sample(connected=connected)
            data.append((tokens, label))

        return data

In [ ]:
class PathfinderDataset(Dataset):
    """PyTorch Dataset for Pathfinder/Path-X tasks."""
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

#### 4.5.1 Pathfinder: Visual path detection in synthetic images

Can the model trace a curvy line through noise to see if two dots are connected?

In [ ]:
# Initialize generator for Pathfinder (32x32)
gen_pathfinder = PathfinderGenerator(
    resolution=32,
    seed=42,
    device=device,
    contour_length=14,
    num_distractor_snakes=6
)

print(f"Using device: {gen_pathfinder.device}")
print(f"Resolution: {gen_pathfinder.resolution}x{gen_pathfinder.resolution}")
print(f"Sequence length: {gen_pathfinder.seq_len}")
print(f"Vocab size: {gen_pathfinder.vocab_size}")
print(f"Classes: 2 (connected / not connected)\n")

# Generate datasets
# LRA uses: 160k train, 20k val, 20k test
train_data_pf = gen_pathfinder.generate(160000)
val_data_pf = gen_pathfinder.generate(20000)
test_data_pf = gen_pathfinder.generate(20000)

# Create PyTorch datasets
train_ds_pf = PathfinderDataset(train_data_pf, seq_len=gen_pathfinder.seq_len)
val_ds_pf = PathfinderDataset(val_data_pf, seq_len=gen_pathfinder.seq_len)
test_ds_pf = PathfinderDataset(test_data_pf, seq_len=gen_pathfinder.seq_len)

# Save Pathfinder
pathfinder_save_path = os.path.join(drive_dir, 'pathfinder_lra.pt')
torch.save({
    'train': train_ds_pf,
    'val': val_ds_pf,
    'test': test_ds_pf,
    'vocab_size': gen_pathfinder.vocab_size,
    'seq_len': gen_pathfinder.seq_len,
    'resolution': gen_pathfinder.resolution,
    'num_classes': gen_pathfinder.num_classes,
    'task': 'pathfinder'
}, pathfinder_save_path)

print(f"\n" + "-"*40)
print("PATHFINDER SAVED")
print("-"*40)
print(f"Saved to: {pathfinder_save_path}")
print(f"Train: {len(train_ds_pf):,} | Val: {len(val_ds_pf):,} | Test: {len(test_ds_pf):,}")

Using device: cpu
Resolution: 32x32
Sequence length: 1024
Vocab size: 257
Classes: 2 (connected / not connected)

Generating 160000 samples (resolution=32x32)...


Generating:   0%|          | 0/160000 [00:00<?, ?it/s]/tmp/ipython-input-826893409.py:65: RuntimeWarning: overflow encountered in scalar add
  img[py, px] = min(255, img[py, px] + int(128 * self.snake_contrast))
Generating:   0%|          | 258/160000 [00:00<02:04, 1285.42it/s]/tmp/ipython-input-826893409.py:65: RuntimeWarning: overflow encountered in scalar add
  img[py, px] = min(255, img[py, px] + int(128 * self.snake_contrast))
Generating:   0%|          | 387/160000 [00:00<02:08, 1242.56it/s]/tmp/ipython-input-826893409.py:65: RuntimeWarning: overflow encountered in scalar add
  img[py, px] = min(255, img[py, px] + int(128 * self.snake_contrast))
Generating:   0%|          | 512/160000 [00:00<02:15, 1175.02it/s]/tmp/ipython-input-826893409.py:65: RuntimeWarning: overflow encountered in scalar add
  img[py, px] = min(255, img[py, px] + int(128 * self.snake_contrast))
Generating:  36%|███▋      | 58134/160000 [00:54<37:03, 45.81it/s]  /tmp/ipython-input-826893409.py:65: RuntimeWarni

Generating 20000 samples (resolution=32x32)...


Generating: 100%|██████████| 20000/20000 [00:15<00:00, 1265.52it/s]


Generating 20000 samples (resolution=32x32)...


Generating: 100%|██████████| 20000/20000 [00:15<00:00, 1265.32it/s]



----------------------------------------
PATHFINDER SAVED
----------------------------------------
Saved to: /content/drive/MyDrive/LRA_Datasets/pathfinder_lra.pt
Train: 160,000 | Val: 20,000 | Test: 20,000


#### 4.5.2 Path-X: Extended version of Pathfinder with 16K sequence length

Same as Pathfinder but much harder because the sequence is 16 times longer.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Google Drive already mounted.")

# Define the directory in Google Drive
drive_dir = "/content/drive/MyDrive/LRA_Datasets"
os.makedirs(drive_dir, exist_ok=True)
print(f"Ensured directory exists: {drive_dir}")

# Initialize generator for Path-X (128x128)
gen_pathx = PathfinderGenerator(
    resolution=128,
    seed=42,
    device=device,
    contour_length=48,  # Longer paths for larger images
    num_distractor_snakes=12  # More distractors
)

print(f"Using device: {gen_pathx.device}")
print(f"Resolution: {gen_pathx.resolution}x{gen_pathx.resolution}")
print(f"Sequence length: {gen_pathx.seq_len}")
print(f"Vocab size: {gen_pathx.vocab_size}")
print(f"Classes: 2 (connected / not connected)\n")

# Generate datasets
train_data_px = gen_pathx.generate(160000)
val_data_px = gen_pathx.generate(20000)
test_data_px = gen_pathx.generate(20000)

# Create PyTorch datasets
train_ds_px = PathfinderDataset(train_data_px, seq_len=gen_pathx.seq_len)
val_ds_px = PathfinderDataset(val_data_px, seq_len=gen_pathx.seq_len)
test_ds_px = PathfinderDataset(test_data_px, seq_len=gen_pathx.seq_len)

# Save Path-X
pathx_save_path = os.path.join(drive_dir, 'pathx_lra.pt')
torch.save({
    'train': train_ds_px,
    'val': val_ds_px,
    'test': test_ds_px,
    'vocab_size': gen_pathx.vocab_size,
    'seq_len': gen_pathx.seq_len,
    'resolution': gen_pathx.resolution,
    'num_classes': gen_pathx.num_classes,
    'task': 'pathx'
}, pathx_save_path)

print(f"\n" + "-"*40)
print("PATH-X SAVED")
print("-"*40)
print(f"Saved to: {pathx_save_path}")
print(f"Train: {len(train_ds_px):,} | Val: {len(val_ds_px):,} | Test: {len(test_ds_px):,}")

Google Drive already mounted.
Ensured directory exists: /content/drive/MyDrive/LRA_Datasets
Using device: cpu
Resolution: 128x128
Sequence length: 16384
Vocab size: 257
Classes: 2 (connected / not connected)

Generating 160000 samples (resolution=128x128)...


Generating:   0%|          | 0/160000 [00:00<?, ?it/s]/tmp/ipython-input-826893409.py:65: RuntimeWarning: overflow encountered in scalar add
  img[py, px] = min(255, img[py, px] + int(128 * self.snake_contrast))
Generating: 100%|██████████| 160000/160000 [14:03<00:00, 189.60it/s]


Generating 20000 samples (resolution=128x128)...


Generating: 100%|██████████| 20000/20000 [01:44<00:00, 190.54it/s]


Generating 20000 samples (resolution=128x128)...


Generating: 100%|██████████| 20000/20000 [01:44<00:00, 190.62it/s]



----------------------------------------
PATH-X SAVED
----------------------------------------
Saved to: /content/drive/MyDrive/LRA_Datasets/pathx_lra.pt
Train: 160,000 | Val: 20,000 | Test: 20,000


## 5. Running all the AI models on the benchmarks

### 5.1 Running all models in ListOps: Hierarchical parsing of mathematical expressions

#### 5.1.1 Running baselines

#### 5.1.2 Running models developed

### 5.2 Running all models in Text: Sentiment classification on IMDb reviews

#### 5.2.1 Running baselines

#### 5.2.2 Running models developed

### 5.3 Running all models in Retrieval: Document similarity matching (ACL Anthology)

#### 5.3.1 Running baselines

#### 5.3.2 Running models developed

### 5.4 Image: Sequential CIFAR-10 classification (pixels as sequence)

#### 5.4.1 Running baselines

#### 5.4.2 Running models developed

### 5.5 Pathfinder: Visual path detection in synthetic images

#### 5.5.1 Running baselines

#### 5.5.2 Running models developed

### 5.6 Path-X: Extended version of Pathfinder with 16K sequence length

#### 5.6.1 Running baselines

#### 5.6.2 Running models developed

## 6. Displaying results based on values obtained

In this section we calculate all the metrics and graphs related to the project

###############################################################################
# CELL 1  — Install packages  (Section 1)
###############################################################################
# !pip install uv
# !uv pip install datasets huggingface_hub lightning numpy pandas pyarrow torch tqdm wandb

###############################################################################
# CELL 2  — Imports  (Section 2)
###############################################################################
import pickle, random, sys, os, re, math, time, json
from concurrent.futures import ThreadPoolExecutor, as_completed

import lightning as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from datasets import load_dataset
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

###############################################################################
# CELL 3  — Baseline LSTM  (Section 3.2)
###############################################################################
class BaselineLSTM(nn.Module):
    """Vanilla LSTM for sequence classification / matching."""
    def __init__(self, vocab_size, d_model=128, n_layers=2, n_classes=2,
                 max_len=2048, dropout=0.1, bidirectional=False, task='classification'):
        super().__init__()
        self.task = task
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        self.lstm = nn.LSTM(
            d_model, d_model, num_layers=n_layers,
            batch_first=True, dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        h_dim = d_model * (2 if bidirectional else 1)
        if task == 'retrieval':
            self.classifier = nn.Sequential(
                nn.Linear(h_dim * 2, d_model),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, n_classes),
            )
        else:
            self.classifier = nn.Sequential(
                nn.Linear(h_dim, d_model),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, n_classes),
            )
        self.dropout = nn.Dropout(dropout)

    def _encode(self, x):
        seq_len = x.size(1)
        h = self.embedding(x) + self.pos_encoding[:, :seq_len, :]
        h = self.dropout(h)
        out, _ = self.lstm(h)
        return out[:, -1, :]                       # last hidden state

    def forward(self, x, x2=None):
        if self.task == 'retrieval' and x2 is not None:
            h1 = self._encode(x)
            h2 = self._encode(x2)
            h = torch.cat([h1, h2], dim=-1)
        else:
            h = self._encode(x)
        return self.classifier(h)


###############################################################################
# CELL 4  — Contextual Memory System (CMS) Module
###############################################################################
class ContextualMemorySystem(nn.Module):
    """
    External memory bank with content-based read / write heads.
    Augments any recurrent model by providing persistent contextual storage
    that the hidden state can attend to at every time step.

    Memory shape: (batch, n_slots, d_mem)
    """
    def __init__(self, d_model, n_slots=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_slots = n_slots
        self.d_model = d_model
        self.n_heads = n_heads
        assert d_model % n_heads == 0
        self.head_dim = d_model // n_heads

        # Read head — multi-head content-based attention
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.read_out = nn.Linear(d_model, d_model)

        # Write head — gated update
        self.write_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid(),
        )
        self.write_content = nn.Linear(d_model, d_model)
        self.erase_gate = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Sigmoid(),
        )

        # Addressing — sharpened softmax over slots
        self.address_proj = nn.Linear(d_model, d_model)
        self.sharpening_beta = nn.Parameter(torch.ones(1))

        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _init_memory(self, batch_size, device):
        """Initialise memory slots with small random values."""
        return torch.randn(batch_size, self.n_slots, self.d_model,
                           device=device) * 0.01

    def _address(self, query, memory):
        """Content-based addressing: softmax similarity to memory keys."""
        # query: (B, D)  memory: (B, S, D)
        k = self.address_proj(memory)                          # (B, S, D)
        score = torch.einsum('bd,bsd->bs', query, k)          # (B, S)
        score = score / (self.d_model ** 0.5)
        beta = F.softplus(self.sharpening_beta)
        weights = F.softmax(beta * score, dim=-1)              # (B, S)
        return weights

    def read(self, h, memory):
        """Multi-head content-based read from memory."""
        B = h.size(0)
        q = self.q_proj(h).view(B, 1, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(memory).view(B, self.n_slots, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(memory).view(B, self.n_slots, self.n_heads, self.head_dim).transpose(1, 2)

        attn = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        read_vec = torch.matmul(attn, v)                      # (B, H, 1, Dh)
        read_vec = read_vec.transpose(1, 2).reshape(B, self.d_model)
        return self.read_out(read_vec)

    def write(self, h, memory):
        """Gated erase-then-add write to memory."""
        weights = self._address(h, memory).unsqueeze(-1)       # (B, S, 1)
        erase = self.erase_gate(h).unsqueeze(1)                # (B, 1, D)
        add = self.write_content(h).unsqueeze(1)               # (B, 1, D)
        memory = memory * (1 - weights * erase) + weights * add
        return memory

    def forward(self, h, memory):
        """
        h      : current hidden state  (B, D)
        memory : memory bank            (B, S, D)
        Returns: (read_vector, updated_memory)
        """
        read_vec = self.read(h, memory)
        # Gate how much of the read vector to incorporate
        gate = self.write_gate(torch.cat([h, read_vec], dim=-1))
        read_vec = gate * read_vec
        read_vec = self.layer_norm(read_vec)
        # Write current hidden state into memory
        memory = self.write(h, memory)
        return read_vec, memory


###############################################################################
# CELL 5  — CMS-Augmented LSTM
###############################################################################
class CMS_LSTM(nn.Module):
    """
    LSTM augmented with Contextual Memory System.
    At each time step the LSTM hidden state queries external memory,
    receives a context vector, and writes back — enabling long-range
    dependency capture beyond standard hidden-state capacity.
    """
    def __init__(self, vocab_size, d_model=128, n_layers=2, n_classes=2,
                 max_len=2048, n_mem_slots=64, n_mem_heads=4,
                 dropout=0.1, task='classification'):
        super().__init__()
        self.task = task
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

        # Core LSTM
        self.lstm_cell = nn.LSTMCell(d_model, d_model)
        self.extra_layers = nn.ModuleList([
            nn.LSTMCell(d_model, d_model) for _ in range(n_layers - 1)
        ])

        # CMS module
        self.cms = ContextualMemorySystem(
            d_model, n_slots=n_mem_slots,
            n_heads=n_mem_heads, dropout=dropout,
        )

        # Fusion gate: blend LSTM output with CMS read vector
        self.fusion_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid(),
        )
        self.fusion_proj = nn.Linear(d_model * 2, d_model)

        if task == 'retrieval':
            self.classifier = nn.Sequential(
                nn.Linear(d_model * 2, d_model),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, n_classes),
            )
        else:
            self.classifier = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, n_classes),
            )
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers

    def _encode(self, x):
        B, T = x.size()
        emb = self.embedding(x) + self.pos_encoding[:, :T, :]
        emb = self.dropout(emb)

        # Initialise LSTM states
        h = [torch.zeros(B, self.d_model, device=x.device) for _ in range(self.n_layers)]
        c = [torch.zeros(B, self.d_model, device=x.device) for _ in range(self.n_layers)]
        # Initialise CMS memory
        memory = self.cms._init_memory(B, x.device)

        # Step through sequence
        for t in range(T):
            inp = emb[:, t, :]

            # Layer 0
            h[0], c[0] = self.lstm_cell(inp, (h[0], c[0]))

            # Extra layers
            for i, cell in enumerate(self.extra_layers, 1):
                h[i], c[i] = cell(h[i-1], (h[i], c[i]))

            # CMS interaction on top-layer hidden state
            read_vec, memory = self.cms(h[-1], memory)

            # Fuse LSTM output with CMS read
            combined = torch.cat([h[-1], read_vec], dim=-1)
            gate = self.fusion_gate(combined)
            fused = self.fusion_proj(combined)
            h[-1] = gate * fused + (1 - gate) * h[-1]

        return h[-1]

    def forward(self, x, x2=None):
        if self.task == 'retrieval' and x2 is not None:
            h1 = self._encode(x)
            h2 = self._encode(x2)
            h = torch.cat([h1, h2], dim=-1)
        else:
            h = self._encode(x)
        return self.classifier(h)


###############################################################################
# CELL 6  — LRA Dataset Generators  (Section 4)
###############################################################################
# ---- 4.1  ListOps ----
class ListOpsGenerator:
    """Hierarchical parsing of mathematical expressions.
    Can the model understand nested structure and apply operations
    in the right order?"""
    def __init__(self, max_depth=10, max_args=10, seed=42, device=None):
        random.seed(seed); np.random.seed(seed)
        self.max_depth = max_depth
        self.max_args = max_args
        self.ops = ['MAX', 'MIN', 'MED', 'SM']
        self.device = device or torch.device('cpu')
        self.vocab = self._build_vocab()
        self.vocab_size = len(self.vocab)

    def _build_vocab(self):
        tokens = ['[PAD]', '[UNK]', '(', ')'] + self.ops + [str(i) for i in range(10)]
        return {t: i for i, t in enumerate(tokens)}

    def _generate_expr(self, depth=0):
        if depth >= self.max_depth or (depth > 0 and random.random() < 0.3):
            val = random.randint(0, 9)
            return str(val), val
        op = random.choice(self.ops)
        n_args = random.randint(2, min(self.max_args, 5))
        children_str, children_val = [], []
        for _ in range(n_args):
            s, v = self._generate_expr(depth + 1)
            children_str.append(s); children_val.append(v)
        if op == 'MAX':   result = max(children_val)
        elif op == 'MIN': result = min(children_val)
        elif op == 'MED': result = int(np.median(children_val))
        elif op == 'SM':  result = sum(children_val) % 10
        expr = f"( {op} " + " ".join(children_str) + " )"
        return expr, result

    def _tokenize(self, expr, max_len=2000):
        tokens = expr.replace('(', '( ').replace(')', ' )').split()
        ids = [self.vocab.get(t, self.vocab['[UNK]']) for t in tokens]
        ids = ids[:max_len]
        ids += [self.vocab['[PAD]']] * (max_len - len(ids))
        return ids

    def generate_dataset(self, n_samples=5000, max_len=2000):
        data, labels = [], []
        for _ in tqdm(range(n_samples), desc='ListOps'):
            expr, val = self._generate_expr()
            data.append(self._tokenize(expr, max_len))
            labels.append(val)
        return torch.tensor(data), torch.tensor(labels)


# ---- 4.2  Text: IMDb Sentiment ----
class IMDbGenerator:
    """Sentiment classification on IMDb reviews.
    Binary classification — can the model capture long-range sentiment?
    Uses byte-level tokenization (vocab size 257)."""
    def __init__(self, max_len=4000, seed=42):
        self.max_len = max_len
        self.seed = seed
        self.vocab_size = 257   # 256 bytes + PAD

    def _byte_tokenize(self, text):
        ids = list(text.encode('utf-8'))[:self.max_len - 1]
        ids = [b + 1 for b in ids]                            # shift: 0 = PAD
        ids += [0] * (self.max_len - len(ids))
        return ids

    def generate_dataset(self, split='train', max_samples=25000):
        ds = load_dataset('imdb', split=split)
        data, labels = [], []
        for item in tqdm(ds, desc=f'IMDb {split}', total=min(len(ds), max_samples)):
            if len(data) >= max_samples:
                break
            data.append(self._byte_tokenize(item['text']))
            labels.append(item['label'])
        return torch.tensor(data), torch.tensor(labels)


# ---- 4.3  Retrieval: ACL Anthology Document Similarity ----
class ACLRetrievalGenerator:
    """Binary classification: are two papers similar (cited) or not?
    Byte-level tokenization (vocab size 257).
    Max sequence length: 4000 per document (8000 total for pair)."""
    def __init__(self, max_len=4000, seed=42):
        self.max_len = max_len
        self.seed = seed
        self.vocab_size = 257

    def _byte_tokenize(self, text):
        ids = list(text.encode('utf-8'))[:self.max_len - 1]
        ids = [b + 1 for b in ids]
        ids += [0] * (self.max_len - len(ids))
        return ids

    def generate_dataset(self, split='train', max_samples=25000):
        try:
            ds = load_dataset('WINGNUS/ACL-OCL', split=split)
        except Exception:
            # Fallback: generate synthetic pairs
            print("⚠️  ACL dataset unavailable, generating synthetic retrieval data.")
            return self._synthetic_dataset(max_samples)
        data_a, data_b, labels = [], [], []
        texts = [(item.get('text',''), item.get('label',0)) for item in ds]
        random.seed(self.seed)
        for i in tqdm(range(min(len(texts)//2, max_samples)), desc='Retrieval'):
            t1, _ = texts[2*i]; t2, _ = texts[2*i+1]
            data_a.append(self._byte_tokenize(t1))
            data_b.append(self._byte_tokenize(t2))
            labels.append(random.randint(0, 1))
        return (torch.tensor(data_a), torch.tensor(data_b)), torch.tensor(labels)

    def _synthetic_dataset(self, n):
        random.seed(self.seed)
        da, db, lb = [], [], []
        for _ in tqdm(range(n), desc='Synth Retrieval'):
            da.append([random.randint(1, 256) for _ in range(self.max_len)])
            db.append([random.randint(1, 256) for _ in range(self.max_len)])
            lb.append(random.randint(0, 1))
        return (torch.tensor(da), torch.tensor(db)), torch.tensor(lb)


# ---- 4.4  Image: Sequential CIFAR-10 ----
class CIFAR10SequentialGenerator:
    """Flatten 32×32×3 image into a length-3072 sequence of pixel values.
    10-class classification. Vocab size 257 (0-255 pixel vals + PAD)."""
    def __init__(self, seed=42):
        self.seed = seed
        self.vocab_size = 257
        self.seq_len = 3072  # 32*32*3

    def generate_dataset(self, split='train', max_samples=50000):
        from torchvision import datasets, transforms
        is_train = split == 'train'
        ds = datasets.CIFAR10(root='./data', train=is_train, download=True)
        data, labels = [], []
        for i in tqdm(range(min(len(ds), max_samples)), desc=f'CIFAR-10 {split}'):
            img, lbl = ds[i]
            arr = np.array(img).flatten()
            ids = (arr + 1).tolist()        # shift: 0 = PAD
            data.append(ids)
            labels.append(lbl)
        return torch.tensor(data, dtype=torch.long), torch.tensor(labels)


# ---- 4.5  Pathfinder (128×128) ----
class PathfinderGenerator:
    """Visual path detection in synthetic images.
    Binary classification: is there a connected path?
    Sequence length: 16384 (128×128 flattened grayscale)."""
    def __init__(self, resolution=128, seed=42):
        self.res = resolution
        self.seq_len = resolution * resolution
        self.vocab_size = 257
        self.seed = seed

    def _generate_sample(self):
        img = np.zeros((self.res, self.res), dtype=np.uint8)
        label = random.randint(0, 1)
        # Draw two dots
        y1, x1 = random.randint(10, 50), random.randint(10, 50)
        y2, x2 = random.randint(70, 117), random.randint(70, 117)
        img[y1-2:y1+2, x1-2:x1+2] = 255
        img[y2-2:y2+2, x2-2:x2+2] = 255
        if label == 1:
            # Draw connecting path with random walk
            cy, cx = y1, x1
            steps = abs(y2-y1) + abs(x2-x1) + random.randint(10, 40)
            for _ in range(steps):
                dy = np.sign(y2 - cy) if random.random() < 0.6 else random.choice([-1,0,1])
                dx = np.sign(x2 - cx) if random.random() < 0.6 else random.choice([-1,0,1])
                cy = np.clip(cy + dy, 0, self.res-1)
                cx = np.clip(cx + dx, 0, self.res-1)
                img[cy, cx] = 200
            img[y2-2:y2+2, x2-2:x2+2] = 255
        # Add noise
        noise = np.random.randint(0, 30, (self.res, self.res), dtype=np.uint8)
        img = np.clip(img.astype(int) + noise, 0, 255).astype(np.uint8)
        ids = (img.flatten() + 1).tolist()   # shift for PAD=0
        return ids, label

    def generate_dataset(self, n_samples=20000):
        random.seed(self.seed); np.random.seed(self.seed)
        data, labels = [], []
        for _ in tqdm(range(n_samples), desc='Pathfinder'):
            ids, lbl = self._generate_sample()
            data.append(ids); labels.append(lbl)
        return torch.tensor(data, dtype=torch.long), torch.tensor(labels)


# ---- 4.6  Path-X (Extended Pathfinder — 16K sequence) ----
class PathXGenerator(PathfinderGenerator):
    """Extended Pathfinder with 128×128 = 16384 sequence length.
    Same task, longer context — the real stress test for long-range models."""
    def __init__(self, seed=42):
        super().__init__(resolution=128, seed=seed)
        # Path-X uses the same 128×128 but with more challenging paths
        self.seq_len = 16384


###############################################################################
# CELL 7  — Generic LRA Dataset wrapper
###############################################################################
class LRADataset(Dataset):
    """Wraps (data, labels) or ((data_a, data_b), labels) for DataLoader."""
    def __init__(self, data, labels):
        self.is_pair = isinstance(data, (tuple, list))
        if self.is_pair:
            self.data_a, self.data_b = data
        else:
            self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if self.is_pair:
            return self.data_a[idx], self.data_b[idx], self.labels[idx]
        return self.data[idx], self.labels[idx]


###############################################################################
# CELL 8  — Training & Evaluation Engine  (Section 5)
###############################################################################
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3,
                is_pair=False, device='cpu', task_name=''):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'{task_name} Epoch {epoch+1}/{epochs}')
        for batch in pbar:
            if is_pair:
                x_a, x_b, y = [b.to(device) for b in batch]
                logits = model(x_a, x_b)
            else:
                x, y = batch[0].to(device), batch[1].to(device)
                logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=f'{total_loss/n_batches:.4f}')
        scheduler.step()

        # Validation
        val_acc = evaluate_model(model, val_loader, is_pair, device)
        history['train_loss'].append(total_loss / n_batches)
        history['val_acc'].append(val_acc)
        best_val_acc = max(best_val_acc, val_acc)
        print(f"  → Val Accuracy: {val_acc:.4f}  (best: {best_val_acc:.4f})")

    return best_val_acc, history


@torch.no_grad()
def evaluate_model(model, loader, is_pair=False, device='cpu'):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        if is_pair:
            x_a, x_b, y = [b.to(device) for b in batch]
            logits = model(x_a, x_b)
        else:
            x, y = batch[0].to(device), batch[1].to(device)
            logits = model(x)
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total if total > 0 else 0.0


###############################################################################
# CELL 9  — Run All Benchmarks  (Section 5.1 – 5.6)
###############################################################################
# ──────────────────────────────────────────────
# Hyperparameters — tune per-task as needed
# ──────────────────────────────────────────────
CONFIGS = {
    'listops':    dict(d_model=128, n_layers=2, max_len=2000, epochs=10, lr=1e-3, bs=32, n_classes=10),
    'imdb':       dict(d_model=128, n_layers=2, max_len=4000, epochs=5,  lr=5e-4, bs=16, n_classes=2),
    'retrieval':  dict(d_model=128, n_layers=2, max_len=4000, epochs=5,  lr=5e-4, bs=8,  n_classes=2),
    'cifar10':    dict(d_model=128, n_layers=2, max_len=3072, epochs=15, lr=1e-3, bs=64, n_classes=10),
    'pathfinder': dict(d_model=64,  n_layers=2, max_len=16384,epochs=10, lr=1e-3, bs=16, n_classes=2),
    'pathx':      dict(d_model=64,  n_layers=2, max_len=16384,epochs=10, lr=1e-3, bs=8,  n_classes=2),
}

results = {}

def run_benchmark(task_key, data_gen_fn, cfg, is_pair=False):
    """Runs baseline LSTM and CMS-LSTM on one LRA task."""
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: {task_key.upper()}")
    print(f"{'='*60}")

    # Generate data
    if is_pair:
        (data_a, data_b), labels = data_gen_fn()
        n = len(labels)
        n_train = int(0.8 * n); n_val = n - n_train
        indices = torch.randperm(n)
        train_ds = LRADataset(
            (data_a[indices[:n_train]], data_b[indices[:n_train]]),
            labels[indices[:n_train]]
        )
        val_ds = LRADataset(
            (data_a[indices[n_train:]], data_b[indices[n_train:]]),
            labels[indices[n_train:]]
        )
        vocab_size = 257
    else:
        data, labels = data_gen_fn()
        n = len(labels)
        n_train = int(0.8 * n); n_val = n - n_train
        indices = torch.randperm(n)
        train_ds = LRADataset(data[indices[:n_train]], labels[indices[:n_train]])
        val_ds   = LRADataset(data[indices[n_train:]], labels[indices[n_train:]])
        vocab_size = data.max().item() + 1 if not is_pair else 257

    train_loader = DataLoader(train_ds, batch_size=cfg['bs'], shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg['bs'], shuffle=False,
                              num_workers=2, pin_memory=True)

    task_type = 'retrieval' if is_pair else 'classification'

    # ---- 5.x.1  Baseline LSTM ----
    print(f"\n--- Baseline LSTM ---")
    baseline = BaselineLSTM(
        vocab_size=vocab_size, d_model=cfg['d_model'],
        n_layers=cfg['n_layers'], n_classes=cfg['n_classes'],
        max_len=cfg['max_len'], task=task_type,
    )
    print(f"Params: {sum(p.numel() for p in baseline.parameters()):,}")
    bl_acc, bl_hist = train_model(
        baseline, train_loader, val_loader,
        epochs=cfg['epochs'], lr=cfg['lr'],
        is_pair=is_pair, device=device, task_name=f'{task_key}-baseline',
    )

    # ---- 5.x.2  CMS-Augmented LSTM ----
    print(f"\n--- CMS-LSTM ---")
    cms_model = CMS_LSTM(
        vocab_size=vocab_size, d_model=cfg['d_model'],
        n_layers=cfg['n_layers'], n_classes=cfg['n_classes'],
        max_len=cfg['max_len'], n_mem_slots=64, n_mem_heads=4,
        task=task_type,
    )
    print(f"Params: {sum(p.numel() for p in cms_model.parameters()):,}")
    cms_acc, cms_hist = train_model(
        cms_model, train_loader, val_loader,
        epochs=cfg['epochs'], lr=cfg['lr'],
        is_pair=is_pair, device=device, task_name=f'{task_key}-CMS',
    )

    results[task_key] = {
        'baseline_acc': bl_acc, 'cms_acc': cms_acc,
        'baseline_hist': bl_hist, 'cms_hist': cms_hist,
        'improvement': cms_acc - bl_acc,
    }
    print(f"\n✅ {task_key}: Baseline={bl_acc:.4f}  CMS={cms_acc:.4f}"
          f"  Δ={cms_acc - bl_acc:+.4f}")
    return results[task_key]


# ==================== 5.1  ListOps ====================
listops_gen = ListOpsGenerator(seed=42, device=device)
run_benchmark('listops',
    lambda: listops_gen.generate_dataset(n_samples=10000, max_len=2000),
    CONFIGS['listops'])

# ==================== 5.2  Text (IMDb) ====================
imdb_gen = IMDbGenerator(max_len=4000, seed=42)
run_benchmark('imdb',
    lambda: imdb_gen.generate_dataset(split='train', max_samples=25000),
    CONFIGS['imdb'])

# ==================== 5.3  Retrieval (ACL) ====================
acl_gen = ACLRetrievalGenerator(max_len=4000, seed=42)
run_benchmark('retrieval',
    lambda: acl_gen.generate_dataset(split='train', max_samples=10000),
    CONFIGS['retrieval'], is_pair=True)

# ==================== 5.4  Image (CIFAR-10) ====================
cifar_gen = CIFAR10SequentialGenerator(seed=42)
run_benchmark('cifar10',
    lambda: cifar_gen.generate_dataset(split='train', max_samples=50000),
    CONFIGS['cifar10'])

# ==================== 5.5  Pathfinder ====================
path_gen = PathfinderGenerator(resolution=128, seed=42)
run_benchmark('pathfinder',
    lambda: path_gen.generate_dataset(n_samples=20000),
    CONFIGS['pathfinder'])

# ==================== 5.6  Path-X (16K) ====================
pathx_gen = PathXGenerator(seed=42)
run_benchmark('pathx',
    lambda: pathx_gen.generate_dataset(n_samples=10000),
    CONFIGS['pathx'])


###############################################################################
# CELL 10  — Results Table & Visualization  (Section 6)
###############################################################################
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("  FINAL RESULTS — LSTM vs CMS-LSTM on Long Range Arena")
print("="*70)
print(f"{'Task':<15} {'Baseline':>10} {'CMS-LSTM':>10} {'Δ':>10}")
print("-"*50)
avg_bl, avg_cms = 0, 0
for task, res in results.items():
    bl = res['baseline_acc']; cm = res['cms_acc']
    avg_bl += bl; avg_cms += cm
    print(f"{task:<15} {bl:>10.4f} {cm:>10.4f} {cm-bl:>+10.4f}")
n = len(results)
print("-"*50)
print(f"{'AVERAGE':<15} {avg_bl/n:>10.4f} {avg_cms/n:>10.4f} {(avg_cms-avg_bl)/n:>+10.4f}")
print("="*70)

# --- Bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

tasks = list(results.keys())
bl_accs = [results[t]['baseline_acc'] for t in tasks]
cms_accs = [results[t]['cms_acc'] for t in tasks]

x = np.arange(len(tasks))
w = 0.35
axes[0].bar(x - w/2, bl_accs, w, label='Baseline LSTM', color='#4C72B0')
axes[0].bar(x + w/2, cms_accs, w, label='CMS-LSTM', color='#DD8452')
axes[0].set_xticks(x); axes[0].set_xticklabels(tasks, rotation=30)
axes[0].set_ylabel('Accuracy'); axes[0].set_title('LRA Benchmark: LSTM vs CMS-LSTM')
axes[0].legend(); axes[0].set_ylim(0, 1.05)

# --- Training curves (val acc) for each task ---
for task in tasks:
    hist = results[task]
    axes[1].plot(hist['cms_hist']['val_acc'], label=f'{task} (CMS)', linewidth=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
axes[1].set_title('CMS-LSTM Validation Accuracy per Task')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lra_lstm_vs_cms_results.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Improvement waterfall ---
fig2, ax2 = plt.subplots(figsize=(10, 5))
improvements = [results[t]['improvement'] * 100 for t in tasks]
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in improvements]
ax2.bar(tasks, improvements, color=colors)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_ylabel('Accuracy Improvement (%)'); ax2.set_title('CMS Improvement over Baseline LSTM')
for i, v in enumerate(improvements):
    ax2.text(i, v + 0.3, f'{v:+.2f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('cms_improvement_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Plots saved to: lra_lstm_vs_cms_results.png, cms_improvement_waterfall.png")

# Task
Integrate Mamba models (baseline and CMS-augmented) into the existing Long Range Arena (LRA) benchmark framework. This involves implementing the Mamba core module, defining `BaselineMamba` and `CMS_Mamba` classes, updating benchmark configurations, modifying the `run_benchmark` function to include Mamba models, executing the benchmarks, and updating the results display to compare Mamba models with LSTMs across all LRA tasks. This will require adding new code cells for the Mamba model definitions and modifying existing cells for configurations, benchmarking logic, and results visualization.

## Implement Mamba Core Module

### Subtask:
Define the MambaBlock or Mamba core computational unit (e.g., S6 layer, SSM) as a separate class within the model definition section (Section 3). This will be a foundational component for both baseline and CMS-augmented Mamba models. This will involve adding new code cells.


**Reasoning**:
The subtask requires defining the `MambaBlock` class, which encapsulates the core Mamba computational unit. This class will include methods for initialization, setting up discrete state-space parameters, and implementing the forward pass. This class needs to be placed in a new code cell immediately following the `ContextualMemorySystem` class definition.



In [ ]:
###############################################################################
# CELL 6  — Mamba Core Computational Unit
###############################################################################
class MambaBlock(nn.Module):
    """The core Mamba computational unit, inspired by the S6 layer and SSM.

    Encapsulates the selective scan mechanism, including linear projections,
    convolution, activation, and the state-space model (SSM) computations.
    """
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4,
                 expand_factor: int = 2, dt_rank: int = 'auto',
                 conv_bias: bool = True, bias: bool = False):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand_factor = expand_factor
        self.d_inner = int(self.expand_factor * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16) if dt_rank == 'auto' else dt_rank

        # Linear projection from d_model to 2 * d_inner (for x and z)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=bias)

        # Conv1D layer
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            groups=self.d_inner, # Depthwise conv
            padding=d_conv - 1, # Ensures output length is same as input
            bias=conv_bias,
        )

        self.activation = nn.SiLU()

        # Linear projection for delta, B, C (d_inner to dt_rank, d_state, d_state respectively)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True) # Bias should be true here

        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=bias)

        # State space parameters A, B, C, D
        self._setup_state_space()

    def _setup_state_space(self):
        # A matrix (d_state, d_inner) - initialized for stability
        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A)) # A is state dependent

        # D vector (d_inner) - feedthrough
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # Initializing dt_projection bias to be positive
        # As per Mamba implementation, it's often initialized in log-space
        nn.init.constant_(self.dt_proj.bias, 0.0) # Or specific large value like -5.0 for stability

    def selective_scan(self, x, delta, A, B, C, D):
        # x: (B, L, d_inner)  delta: (B, L, d_inner)  A: (d_inner, d_state)  B: (B, L, d_state) C: (B, L, d_state) D: (d_inner)
        # A, B, C, D are effectively expanded to (B, L, d_inner, d_state) or (B, L, d_state, d_inner)

        # Apply scaling and activation to delta
        delta = F.softplus(self.dt_proj(delta)) # (B, L, d_inner)

        # Discretize A, B
        # dA = exp(delta * A)
        # dB = delta * B
        # This requires broadcasting A and B across d_inner dimension

        # A is (d_inner, d_state), expand delta to (B, L, d_inner, d_state) for element-wise multiplication
        A = -torch.exp(self.A_log.float()) # Ensure A is negative for stability
        dA = torch.exp(delta.unsqueeze(-1) * A) # (B, L, d_inner, d_state)
        dB = delta.unsqueeze(-1) * B.unsqueeze(2) # (B, L, d_inner, d_state)

        # Perform the scan
        h = torch.zeros(x.size(0), self.d_inner, self.d_state, device=x.device) # (B, d_inner, d_state)
        ys = []
        for i in range(x.size(1)): # Iterate over sequence length
            h = dA[:, i, :, :] * h + dB[:, i, :, :] * x[:, i, :].unsqueeze(-1) # (B, d_inner, d_state)
            y = (C[:, i, :, :] @ h.transpose(1,2)).squeeze(2) # (B, d_inner)
            ys.append(y)

        y_ssm = torch.stack(ys, dim=1) # (B, L, d_inner)

        return y_ssm + x * D # Add feedthrough term


    def forward(self, x: torch.Tensor): # x: (B, L, d_model)
        B, L, _ = x.shape

        # Input projection
        xz = self.in_proj(x) # (B, L, 2 * d_inner)
        x, z = xz.chunk(2, dim=-1) # x: (B, L, d_inner), z: (B, L, d_inner)

        # Permute for Conv1D (B, d_inner, L) and apply conv
        x = x.transpose(1, 2) # (B, d_inner, L)
        x = self.conv1d(x)[:, :, :L] # (B, d_inner, L) -- truncate padding
        x = x.transpose(1, 2) # (B, L, d_inner)

        x = self.activation(x) # (B, L, d_inner)

        # Project to get delta, B, C
        # x_proj is (B, L, dt_rank + 2 * d_state)
        x_proj = self.x_proj(x)
        delta, B_proj, C_proj = x_proj.split([self.dt_rank, self.d_state, self.d_state], dim=-1)

        # Expand B_proj and C_proj across d_inner for selective scan
        # B_proj: (B, L, d_state) -> (B, L, d_inner, d_state)
        # C_proj: (B, L, d_state) -> (B, L, d_inner, d_state)
        B = B_proj # In this simplified scan, B and C are directly used. They need to be broadcast implicitly.
        C = C_proj # Real Mamba uses more complex expansion for B and C

        # Perform selective scan
        y = self.selective_scan(x, delta, self.A_log, B, C, self.D)

        # Apply activation to z and multiply with y
        y = y * self.activation(z)

        # Output projection
        out = self.out_proj(y) # (B, L, d_model)

        return out

NameError: name 'nn' is not defined

**Reasoning**:
The previous error 'NameError: name 'nn' is not defined' indicates that the `torch.nn` module was not imported or not accessible in the scope where `MambaBlock` was defined. I will add the necessary imports (`torch`, `torch.nn as nn`, `torch.nn.functional as F`, and `math`) directly to the `MambaBlock` cell to ensure all dependencies are met within its scope, making the cell self-contained for execution.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

###############################################################################
# CELL 6  — Mamba Core Computational Unit
###############################################################################
class MambaBlock(nn.Module):
    """The core Mamba computational unit, inspired by the S6 layer and SSM.

    Encapsulates the selective scan mechanism, including linear projections,
    convolution, activation, and the state-space model (SSM) computations.
    """
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4,
                 expand_factor: int = 2, dt_rank: int = 'auto',
                 conv_bias: bool = True, bias: bool = False):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand_factor = expand_factor
        self.d_inner = int(self.expand_factor * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16) if dt_rank == 'auto' else dt_rank

        # Linear projection from d_model to 2 * d_inner (for x and z)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=bias)

        # Conv1D layer
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            groups=self.d_inner, # Depthwise conv
            padding=d_conv - 1, # Ensures output length is same as input
            bias=conv_bias,
        )

        self.activation = nn.SiLU()

        # Linear projection for delta, B, C (d_inner to dt_rank, d_state, d_state respectively)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True) # Bias should be true here

        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=bias)

        # State space parameters A, B, C, D
        self._setup_state_space()

    def _setup_state_space(self):
        # A matrix (d_state, d_inner) - initialized for stability
        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A)) # A is state dependent

        # D vector (d_inner) - feedthrough
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # Initializing dt_projection bias to be positive
        # As per Mamba implementation, it's often initialized in log-space
        nn.init.constant_(self.dt_proj.bias, 0.0) # Or specific large value like -5.0 for stability

    def selective_scan(self, x, delta, A, B, C, D):
        # x: (B, L, d_inner)  delta: (B, L, d_inner)  A: (d_inner, d_state)  B: (B, L, d_state) C: (B, L, d_state) D: (d_inner)
        # A, B, C, D are effectively expanded to (B, L, d_inner, d_state) or (B, L, d_state, d_inner)

        # Apply scaling and activation to delta
        delta = F.softplus(self.dt_proj(delta)) # (B, L, d_inner)

        # Discretize A, B
        # dA = exp(delta * A)
        # dB = delta * B
        # This requires broadcasting A and B across d_inner dimension

        # A is (d_inner, d_state), expand delta to (B, L, d_inner, d_state) for element-wise multiplication
        A = -torch.exp(self.A_log.float()) # Ensure A is negative for stability
        dA = torch.exp(delta.unsqueeze(-1) * A) # (B, L, d_inner, d_state)
        dB = delta.unsqueeze(-1) * B.unsqueeze(2) # (B, L, d_inner, d_state)

        # Perform the scan
        h = torch.zeros(x.size(0), self.d_inner, self.d_state, device=x.device) # (B, d_inner, d_state)
        ys = []
        for i in range(x.size(1)): # Iterate over sequence length
            h = dA[:, i, :, :] * h + dB[:, i, :, :] * x[:, i, :].unsqueeze(-1) # (B, d_inner, d_state)
            y = (C[:, i, :, :] @ h.transpose(1,2)).squeeze(2) # (B, d_inner)
            ys.append(y)

        y_ssm = torch.stack(ys, dim=1) # (B, L, d_inner)

        return y_ssm + x * D # Add feedthrough term


    def forward(self, x: torch.Tensor): # x: (B, L, d_model)
        B, L, _ = x.shape

        # Input projection
        xz = self.in_proj(x) # (B, L, 2 * d_inner)
        x, z = xz.chunk(2, dim=-1) # x: (B, L, d_inner), z: (B, L, d_inner)

        # Permute for Conv1D (B, d_inner, L) and apply conv
        x = x.transpose(1, 2) # (B, d_inner, L)
        x = self.conv1d(x)[:, :, :L] # (B, d_inner, L) -- truncate padding
        x = x.transpose(1, 2) # (B, L, d_inner)

        x = self.activation(x) # (B, L, d_inner)

        # Project to get delta, B, C
        # x_proj is (B, L, dt_rank + 2 * d_state)
        x_proj = self.x_proj(x)
        delta, B_proj, C_proj = x_proj.split([self.dt_rank, self.d_state, self.d_state], dim=-1)

        # Expand B_proj and C_proj across d_inner for selective scan
        # B_proj: (B, L, d_state) -> (B, L, d_inner, d_state)
        # C_proj: (B, L, d_state) -> (B, L, d_inner, d_state)
        B = B_proj # In this simplified scan, B and C are directly used. They need to be broadcast implicitly.
        C = C_proj # Real Mamba uses more complex expansion for B and C

        # Perform selective scan
        y = self.selective_scan(x, delta, self.A_log, B, C, self.D)

        # Apply activation to z and multiply with y
        y = y * self.activation(z)

        # Output projection
        out = self.out_proj(y) # (B, L, d_model)

        return out